# Neo4j graph explorer for Copilot-assisted prompting

This notebook connects to the local Neo4j Desktop database at:

- C:/Users/prathikak/.Neo4jDesktop2/Data/dbmss/dbms-53ca4fd9-2306-4c45-a7bd-a4188d584db2

It lets you:

- test the Neo4j connection
- inspect labels and relationship types
- run Cypher queries
- render the result as an interactive graph
- filter the graph by labels, relationship types, and row limit

Copilot Chat itself cannot be called directly from Python, but this notebook is set up for a prompt -> Cypher -> query -> visualize workflow. Use Copilot Chat to draft Cypher, then paste it into the manual query cell near the end.

In [ ]:
c:\Users\prathikak\Downloads\neo4j-query-studio

In [ ]:
import sys
import subprocess

for pkg in ["neo4j", "pandas", "pyvis", "networkx", "ipywidgets"]:
    try:
        __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

print("Dependencies ready.")

In [ ]:
from pathlib import Path
import html
import os
import pandas as pd
import ipywidgets as widgets
from neo4j import GraphDatabase
from pyvis.network import Network
from IPython.display import HTML, clear_output, display

DB_PATH = Path("C:/Users/prathikak/.Neo4jDesktop2/Data/dbmss/dbms-53ca4fd9-2306-4c45-a7bd-a4188d584db2")
BOLT_URI = os.getenv("NEO4J_BOLT_URI", "bolt://localhost:7687")
USERNAME = os.getenv("NEO4J_USER", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD")

def get_driver():
    if PASSWORD:
        return GraphDatabase.driver(BOLT_URI, auth=(USERNAME, PASSWORD))
    return GraphDatabase.driver(BOLT_URI)

driver = get_driver()
try:
    driver.verify_connectivity()
    print(f"Connected to {BOLT_URI} as {USERNAME}.")
except Exception as exc:
    print("Connection test failed. Check the Bolt URI, username, and password.")
    raise exc

In [ ]:
def run_table_query(cypher, params=None):
    params = params or {}
    with driver.session(database="neo4j") as session:
        rows = [record.data() for record in session.run(cypher, params)]
    return pd.DataFrame(rows)

def run_graph_query(cypher, params=None):
    params = params or {}
    with driver.session(database="neo4j") as session:
        return list(session.run(cypher, params))

def get_schema():
    labels = run_table_query("CALL db.labels() YIELD label RETURN label ORDER BY label")["label"].tolist()
    rel_types = run_table_query("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType")["relationshipType"].tolist()
    properties = run_table_query("CALL db.propertyKeys() YIELD propertyKey RETURN propertyKey ORDER BY propertyKey")["propertyKey"].tolist()
    return {"labels": labels, "relationship_types": rel_types, "property_keys": properties}

def fetch_graph(node_labels=None, rel_types=None, limit=150):
    node_labels = node_labels or []
    rel_types = rel_types or []
    cypher = """
    MATCH (n)-[r]-(m)
    WHERE ($node_labels = [] OR any(lbl IN labels(n) WHERE lbl IN $node_labels) OR any(lbl IN labels(m) WHERE lbl IN $node_labels))
      AND ($rel_types = [] OR type(r) IN $rel_types)
    RETURN n, r, m
    LIMIT $limit
    """
    return run_graph_query(cypher, {"node_labels": node_labels, "rel_types": rel_types, "limit": int(limit)})

def node_caption(node):
    for key in ["name", "title", "label", "id", "uuid", "code"]:
        if key in node and node[key] not in (None, ""):
            return str(node[key])
    return node.element_id

PALETTE = ["#38bdf8", "#f0abfc", "#6ee7b7", "#f59e0b", "#fb7185", "#a78bfa", "#34d399", "#f472b6"]

def visualize_graph(records, color_map=None):
    color_map = color_map or {}
    net = Network(height="780px", width="100%", bgcolor="#ffffff", font_color="#1f2937", notebook=True, cdn_resources="in_line")
    seen = set()
    for record in records:
        n = record["n"]
        r = record["r"]
        m = record["m"]
        for node in (n, m):
            node_id = node.element_id
            if node_id in seen:
                continue
            labels = list(node.labels)
            primary_label = labels[0] if labels else "Node"
            caption = html.escape(node_caption(node))
            properties = "<br>".join(f"{html.escape(str(k))}: {html.escape(str(v))}" for k, v in node.items())
            title = f"<b>{html.escape(primary_label)}</b><br>{caption}" + (f"<br>{properties}" if properties else "")
            net.add_node(
                node_id,
                label=caption,
                title=title,
                group=primary_label,
                color=color_map.get(primary_label, "#38bdf8"),
                value=max(len(node), 1),
                shape="dot",
            )
            seen.add(node_id)
        net.add_edge(n.element_id, m.element_id, label=r.type, title=r.type)
    net.toggle_physics(True)
    return net.generate_html()

schema = get_schema()
schema

In [ ]:
color_map = {label: PALETTE[i % len(PALETTE)] for i, label in enumerate(schema["labels"])}

label_picker = widgets.SelectMultiple(
    options=schema["labels"],
    description="Labels",
    layout=widgets.Layout(width="420px", height="160px")
)
rel_picker = widgets.SelectMultiple(
    options=schema["relationship_types"],
    description="Rels",
    layout=widgets.Layout(width="420px", height="160px")
)
limit_slider = widgets.IntSlider(value=150, min=10, max=1000, step=10, description="Limit")
render_button = widgets.Button(description="Render graph", button_style="primary")
output = widgets.Output()

def on_render(_):
    with output:
        clear_output(wait=True)
        records = fetch_graph(list(label_picker.value), list(rel_picker.value), limit_slider.value)
        if not records:
            print("No rows matched the selected filters.")
            return
        display(HTML(visualize_graph(records, color_map=color_map)))

render_button.on_click(on_render)
display(widgets.VBox([label_picker, rel_picker, limit_slider, render_button, output]))

In [ ]:
# Paste Cypher that Copilot Chat suggests here, then run this cell.
cypher_query = """
MATCH (n)-[r]-(m)
RETURN n, r, m
LIMIT 25
"""

manual_records = run_graph_query(cypher_query)
display(HTML(visualize_graph(manual_records, color_map=color_map)))